# 🦵 RSNA Knee Abnormality Detection — Starter Notebook

## Über diesen Wettbewerb

In diesem Wettbewerb sollen für jede **Study** (eine Knie-MRT-Untersuchung eines Patienten) **12 binäre
Zielvariablen** vorhergesagt werden — jeweils als Konfidenzwert zwischen 0 und 1:

| # | Label | Bedeutung |
|---|-------|-----------|
| 1 | `ACL` | Riss des vorderen Kreuzbands |
| 2 | `MCL` | Verletzung des Innenbands |
| 3 | `Medial_Meniscus` | Riss des medialen Meniskus |
| 4 | `Lateral_Meniscus` | Riss des lateralen Meniskus |
| 5 | `Medial_OA` | Arthrose medial |
| 6 | `Lateral_OA` | Arthrose lateral |
| 7 | `PF_OA` | Patellofemorale Arthrose |
| 8 | `Effusion` | Gelenkerguss |
| 9 | `Synovitis` | Synovialitis (Entzündung der Gelenkinnenhaut) |
| 10 | `Bakers` | Baker-Zyste |
| 11 | `Contusion` | Knochenkontusion (Bone Bruise) |
| 12 | `Fracture` | Fraktur |

> **Hinweis:** Die exakten Spaltennamen können in `train.csv` leicht abweichen (z. B. Groß-/Kleinschreibung,
> Unterstriche). Wir lesen sie deshalb weiter unten **programmatisch** aus `sample_submission.csv` aus,
> anstatt sie hart zu codieren — das macht das Notebook robust gegenüber kleinen Namensänderungen.

## Datenquellen

- **`train.csv`** — 1 Zeile pro Study: `StudyInstanceUID`, `PatientSex`, `Report` (freier, mehrsprachiger
  Radiologie-Bericht) + die 12 Zielspalten.
- **`train_series.csv`** — 1 Zeile pro Serie (mehrere Serien pro Study): `StudyInstanceUID`,
  `SeriesInstanceUID`, `Fluid_Sensitive`, `Fat_Suppression`, `Anatomical_Plane`.
- **`train_series/<StudyInstanceUID>/<SeriesInstanceUID>/<SOPInstanceUID>.dcm`** — die eigentlichen
  DICOM-Bildslices (20–45 pro Serie).
- **`test.csv`, `test_series.csv`, `test_series/`** — Platzhalter-Testdaten (werden beim Scoring durch
  ca. 1300 echte Studies ersetzt).
- **`sample_submission.csv`** — Formatvorlage, alle Werte auf 0.5.

## Metrik

**Macro-averaged AUC-ROC** über alle 12 Labels: für jedes Label wird der AUC separat berechnet, danach
wird der ungewichtete Mittelwert gebildet. Das bedeutet: **jedes Label zählt gleich viel**, unabhängig
davon wie selten oder häufig es ist — seltene Pathologien (z. B. `Fracture`) sind also genauso wichtig
wie häufige.

## Ziel dieses Notebooks

Dieses Notebook soll als **verständlicher Startpunkt** für die Community dienen:

1. Wir explorieren die Daten (Labels, Bilder, Metadaten, Text).
2. Wir bauen eine **bewusst einfache** Baseline (Text-only: TF-IDF + logistische Regression).
3. Wir zeigen als **Bonus** einen Ansatz für Bildmerkmale (CNN-Features pro Serie, aggregiert pro Study).
4. Wir erzeugen eine gültige `submission.csv`.
5. Wir geben konkrete Ideen, wie man von hier aus weitermachen kann.

Alles ist so geschrieben, dass es **unter 9 Stunden** läuft, **ohne Internetzugriff während der
Submission-Ausführung** funktioniert (sofern vortrainierte Gewichte vorab als Kaggle-Dataset eingebunden
werden — dazu mehr im Bonus-Abschnitt) und robust mit fehlenden/leeren Werten umgeht.


## 1. Imports & Setup

Wir importieren alle benötigten Libraries, setzen Seeds für Reproduzierbarkeit und definieren die
Pfade zu den Wettbewerbsdaten. Optionale Libraries (z. B. `pydicom`, `langdetect`, `torch`) werden mit
`try/except` importiert, damit das Notebook auch dann nicht sofort abbricht, wenn eine davon in der
aktuellen Umgebung fehlt — wir weisen dann stattdessen deutlich darauf hin.


In [ ]:
import os
import sys
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

# ---- Optionale Libraries ---------------------------------------------------
try:
    import pydicom
    HAS_PYDICOM = True
except ImportError:
    HAS_PYDICOM = False
    print("⚠️  pydicom nicht verfügbar — DICOM-Bilder können nicht geladen werden. "
          "Mit `pip install pydicom` nachinstallieren.")

try:
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 0  # deterministische Sprach-Erkennung
    HAS_LANGDETECT = True
except ImportError:
    HAS_LANGDETECT = False
    print("⚠️  langdetect nicht verfügbar — Sprach-Verteilung der Reports wird übersprungen. "
          "Mit `pip install langdetect` nachinstallieren.")

try:
    import torch
    import torchvision
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    print("⚠️  torch/torchvision nicht verfügbar — der Bonus-Bildbaseline-Abschnitt wird übersprungen.")

# ---- Reproduzierbarkeit -----------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if HAS_TORCH:
    torch.manual_seed(SEED)

# ---- Pfade -------------------------------------------------------------------
# Kaggle stellt Competition-Daten standardmäßig unter /kaggle/input/<competition-slug>/ bereit.
# Der Slug ist hier als Variable hinterlegt, damit man ihn an einer Stelle anpassen kann, falls
# Kaggle einen abweichenden Ordnernamen verwendet.
COMPETITION_SLUG = "rsna-knee-abnormality-detection"
DATA_DIR = Path(f"/kaggle/input/{COMPETITION_SLUG}")

# Fallback: falls der Pfad lokal nicht existiert (z. B. beim Testen außerhalb von Kaggle),
# versuchen wir ein paar naheliegende Alternativen, bevor wir aufgeben.
if not DATA_DIR.exists():
    candidates = list(Path("/kaggle/input").glob("*knee*")) if Path("/kaggle/input").exists() else []
    if candidates:
        DATA_DIR = candidates[0]
        print(f"ℹ️  Angepasster Datenpfad gefunden: {DATA_DIR}")
    else:
        print(f"⚠️  Datenpfad {DATA_DIR} nicht gefunden. Bitte DATA_DIR manuell anpassen, "
              f"falls dieses Notebook außerhalb von Kaggle ausgeführt wird.")

TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_SERIES_CSV = DATA_DIR / "train_series.csv"
TRAIN_IMG_DIR = DATA_DIR / "train_series"
TEST_CSV = DATA_DIR / "test.csv"
TEST_SERIES_CSV = DATA_DIR / "test_series.csv"
TEST_IMG_DIR = DATA_DIR / "test_series"
SAMPLE_SUB_CSV = DATA_DIR / "sample_submission.csv"

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Datenpfad:", DATA_DIR)
print("Vorhandene Dateien/Ordner:", [p.name for p in DATA_DIR.glob("*")] if DATA_DIR.exists() else "—")


## 2. Explorative Datenanalyse (EDA)

Bevor wir modellieren, verschaffen wir uns einen Überblick über:

- die **Label-Verteilung** (wie unbalanciert ist jede Pathologie?),
- Beispielbilder aus den **DICOM-Serien**,
- die Verteilung der **Serien-Metadaten** (`Anatomical_Plane`, `Fluid_Sensitive`, `Fat_Suppression`),
- Länge und **Sprache** der Radiologie-Berichte,
- **fehlende Werte** über alle Tabellen hinweg.


In [ ]:
train = pd.read_csv(TRAIN_CSV)
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)

print("train.csv shape:", train.shape)
train.head()


In [ ]:
# Zielspalten dynamisch aus sample_submission.csv ableiten (robust gegenüber leichten
# Namensabweichungen) — alle Spalten außer der ID-Spalte gelten als Zielvariablen.
ID_COL = "StudyInstanceUID"
TARGET_COLS = [c for c in sample_sub.columns if c != ID_COL]

print(f"Anzahl Zielvariablen: {len(TARGET_COLS)}")
print("Zielvariablen:", TARGET_COLS)

# Sicherstellen, dass diese Spalten auch tatsächlich in train.csv vorhanden sind.
missing_targets = [c for c in TARGET_COLS if c not in train.columns]
if missing_targets:
    print(f"⚠️  Folgende Zielspalten fehlen in train.csv und müssen manuell abgeglichen werden: {missing_targets}")


In [ ]:
# ---- Label-Verteilung -------------------------------------------------------
label_rates = train[TARGET_COLS].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=label_rates.values, y=label_rates.index, ax=ax, color="steelblue")
ax.set_xlabel("Positiv-Rate (Anteil Study mit Befund = 1)")
ax.set_title("Label-Verteilung über alle 12 Zielvariablen")
for i, v in enumerate(label_rates.values):
    ax.text(v + 0.005, i, f"{v:.1%}", va="center")
plt.tight_layout()
plt.show()

print("\nStark unbalancierte Labels sind für die Macro-AUC besonders kritisch — "
      "ein einzelnes falsch kalibriertes seltenes Label kann den Score stark drücken.")


In [ ]:
# ---- Fehlende Werte ----------------------------------------------------------
missing = train.isna().mean().sort_values(ascending=False)
missing = missing[missing > 0]
if len(missing):
    print("Spalten mit fehlenden Werten in train.csv:")
    print(missing)
else:
    print("Keine fehlenden Werte in train.csv gefunden.")


In [ ]:
# ---- Beispielbilder aus DICOM-Serien -----------------------------------------
def load_dicom_slice(path):
    \"\"\"Lädt einen einzelnen DICOM-Slice und normalisiert ihn auf [0, 1] für die Anzeige.\"\"\"
    dcm = pydicom.dcmread(str(path))
    arr = dcm.pixel_array.astype(np.float32)
    arr -= arr.min()
    if arr.max() > 0:
        arr /= arr.max()
    return arr

if HAS_PYDICOM and TRAIN_IMG_DIR.exists():
    # Wir nehmen die ersten paar Studies und zeigen jeweils den mittleren Slice der ersten Serie.
    example_study_dirs = sorted(TRAIN_IMG_DIR.glob("*"))[:6]

    fig, axes = plt.subplots(2, 3, figsize=(13, 9))
    for ax, study_dir in zip(axes.flat, example_study_dirs):
        series_dirs = sorted(study_dir.glob("*"))
        if not series_dirs:
            continue
        dcm_files = sorted(series_dirs[0].glob("*.dcm"))
        if not dcm_files:
            continue
        mid_file = dcm_files[len(dcm_files) // 2]
        try:
            img = load_dicom_slice(mid_file)
            ax.imshow(img, cmap="gray")
            ax.set_title(study_dir.name[:12] + "…", fontsize=9)
        except Exception as e:
            ax.set_title(f"Fehler: {e}", fontsize=8)
        ax.axis("off")
    plt.suptitle("Beispiel-Slices (mittlerer Slice der ersten Serie je Study)")
    plt.tight_layout()
    plt.show()
else:
    print("Überspringe Bild-Visualisierung (pydicom fehlt oder Bildordner nicht gefunden).")


In [ ]:
# ---- Serien-Metadaten ---------------------------------------------------------
train_series = pd.read_csv(TRAIN_SERIES_CSV)
print("train_series.csv shape:", train_series.shape)
train_series.head()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col in zip(axes, ["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]):
    if col in train_series.columns:
        counts = train_series[col].value_counts(dropna=False)
        sns.barplot(x=counts.index.astype(str), y=counts.values, ax=ax, color="darkorange")
        ax.set_title(col)
        ax.set_ylabel("Anzahl Serien")
        ax.tick_params(axis="x", rotation=30)
    else:
        ax.set_title(f"{col} nicht gefunden")

plt.tight_layout()
plt.show()

# Wie viele Serien hat eine Study im Schnitt?
series_per_study = train_series.groupby(ID_COL).size()
print(f"Serien pro Study — Median: {series_per_study.median():.0f}, "
      f"Min: {series_per_study.min()}, Max: {series_per_study.max()}")


In [ ]:
# ---- Report-Text: Länge und Sprache --------------------------------------------
train["report_len_chars"] = train["Report"].fillna("").str.len()
train["report_len_words"] = train["Report"].fillna("").str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(train["report_len_words"], bins=40, ax=axes[0], color="seagreen")
axes[0].set_title("Verteilung der Report-Länge (Wörter)")
axes[0].set_xlabel("Anzahl Wörter")

if HAS_LANGDETECT:
    # Sprach-Erkennung ist relativ langsam -> auf eine Stichprobe begrenzen für schnelle EDA.
    sample_reports = train["Report"].fillna("").sample(min(500, len(train)), random_state=SEED)

    def safe_detect(text):
        text = text.strip()
        if len(text) < 5:
            return "unknown"
        try:
            return detect(text)
        except Exception:
            return "unknown"

    langs = sample_reports.apply(safe_detect)
    lang_counts = langs.value_counts()
    sns.barplot(x=lang_counts.index, y=lang_counts.values, ax=axes[1], color="mediumpurple")
    axes[1].set_title(f"Sprach-Verteilung (Stichprobe n={len(sample_reports)})")
    axes[1].set_ylabel("Anzahl Reports")
else:
    axes[1].set_title("Sprach-Erkennung übersprungen (langdetect fehlt)")

plt.tight_layout()
plt.show()

print("Die Reports sind mehrsprachig — bei der Textverarbeitung sollten wir daher keine "
      "sprachspezifischen Annahmen (z. B. englische Stopwörter) hart codieren.")


## 3. Baseline-Modell: TF-IDF + Logistische Regression (Text-only)

Wir starten bewusst **einfach**: Nur der freie Radiologie-Bericht (`Report`) wird als Merkmal genutzt.
Das hat mehrere Vorteile für eine Baseline:

- **Schnell** (Training dauert Sekunden bis wenige Minuten, keine GPU nötig).
- **Robust** — keine Abhängigkeit von DICOM-Parsing-Eigenheiten (z. B. unterschiedliche
  Transfer-Syntaxen, fehlende Slices).
- **Leicht nachvollziehbar und erweiterbar** — andere können direkt Bildmerkmale, andere
  Textrepräsentationen (z. B. Embeddings) oder eine Fusion beider Modalitäten ergänzen.

Da die Berichte oft direkt die Diagnose beschreiben (z. B. "Riss des vorderen Kreuzbands"), ist ein
Text-only-Modell überraschend stark — aber natürlich **nicht das Ende der Fahnenstange**, siehe Fazit.

Wir trainieren pro Label ein eigenes binäres Logistische-Regressions-Modell
(`OneVsRestClassifier`), da die 12 Zielvariablen nicht exklusiv sind (ein Patient kann mehrere
Pathologien gleichzeitig haben — Multi-Label, nicht Multi-Class).


In [ ]:
# ---- Train/Validation Split ----------------------------------------------------
# Hinweis: Ein einfacher Random-Split ist für eine Baseline ausreichend. Für ein stärkeres Modell
# lohnt sich ein iterativer Multi-Label-Stratified-Split (siehe Fazit), damit seltene Labels in
# Train und Val ähnlich verteilt sind.
train_df, val_df = train_test_split(train, test_size=0.2, random_state=SEED)

X_train_text = train_df["Report"].fillna("")
X_val_text = val_df["Report"].fillna("")
y_train = train_df[TARGET_COLS].values
y_val = val_df[TARGET_COLS].values

print(f"Train: {len(train_df)} Studies | Val: {len(val_df)} Studies")


In [ ]:
# ---- TF-IDF Vektorisierung -------------------------------------------------------
# max_features begrenzt die Vokabulargröße, um Overfitting und Laufzeit im Griff zu behalten.
# ngram_range=(1, 2) erfasst neben Einzelwörtern auch kurze Wortkombinationen (z. B. "vorderes Kreuzband").
tfidf = TfidfVectorizer(
    max_features=20_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf = tfidf.transform(X_val_text)

print("TF-IDF Feature-Matrix (Train):", X_train_tfidf.shape)


In [ ]:
# ---- Training: 1 Logistische-Regression pro Label --------------------------------
base_clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
clf = OneVsRestClassifier(base_clf, n_jobs=-1)
clf.fit(X_train_tfidf, y_train)

val_proba = clf.predict_proba(X_val_tfidf)


In [ ]:
# ---- Evaluation: Macro-averaged AUC-ROC -------------------------------------------
per_label_auc = {}
for i, col in enumerate(TARGET_COLS):
    # Falls im Val-Split ein Label nur eine Klasse enthält, ist AUC nicht definiert -> überspringen.
    if len(np.unique(y_val[:, i])) < 2:
        per_label_auc[col] = np.nan
        continue
    per_label_auc[col] = roc_auc_score(y_val[:, i], val_proba[:, i])

auc_series = pd.Series(per_label_auc).sort_values(ascending=False)
macro_auc = auc_series.mean()

print("Per-Label AUC-ROC (Validation):")
print(auc_series.round(4))
print(f"\n➡️  Macro-averaged AUC-ROC (Validation): {macro_auc:.4f}")

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=auc_series.values, y=auc_series.index, ax=ax, color="teal")
ax.axvline(0.5, color="gray", linestyle="--", label="Zufalls-Baseline (AUC=0.5)")
ax.axvline(macro_auc, color="red", linestyle="--", label=f"Macro-AUC = {macro_auc:.3f}")
ax.set_xlabel("AUC-ROC")
ax.set_title("Validation AUC-ROC pro Label")
ax.legend()
plt.tight_layout()
plt.show()


## 4. Bonus (optional): Einfache Bildmerkmal-Baseline

Als **Erweiterungsidee** zeigen wir hier, wie man Bildmerkmale statt (oder zusätzlich zu) Text nutzen
könnte: Wir extrahieren pro Serie einen repräsentativen Slice (den mittleren), lassen ihn durch ein
**vortrainiertes CNN** (ResNet-18) laufen und mitteln die resultierenden Feature-Vektoren über alle
Serien einer Study, um pro Study einen einzigen Bildmerkmalsvektor zu erhalten.

> ⚠️ **Wichtig zum Thema "kein Internetzugriff während der Submission":** ImageNet-vortrainierte
> Gewichte von `torchvision` werden normalerweise beim ersten Aufruf aus dem Internet geladen. Für die
> tatsächliche Submission (Internet **aus**) muss man die Gewichte entweder
> 1. vorab in einem Notebook mit Internet herunterladen und als eigenes **Kaggle-Dataset** hochladen,
>    dann in der Submission von dort laden, oder
> 2. ein bereits als Kaggle-Dataset verfügbares vortrainiertes Modell einbinden.
>
> Dieser Abschnitt ist bewusst als **Demonstration/Ausgangspunkt** gehalten und wird standardmäßig
> übersprungen, wenn `torch` nicht verfügbar ist oder keine Bilddaten gefunden werden — die finale
> Submission in diesem Notebook basiert auf der robusten Text-Baseline oben.


In [ ]:
def extract_study_image_features(study_dir, model, transform, device, max_series=4):
    \"\"\"Extrahiert CNN-Features für eine Study, gemittelt über bis zu `max_series` Serien
    (jeweils der mittlere Slice pro Serie).\"\"\"
    series_dirs = sorted(Path(study_dir).glob("*"))[:max_series]
    feats = []
    for series_dir in series_dirs:
        dcm_files = sorted(Path(series_dir).glob("*.dcm"))
        if not dcm_files:
            continue
        mid_file = dcm_files[len(dcm_files) // 2]
        try:
            img = load_dicom_slice(mid_file)  # [0, 1], grayscale
            img_rgb = np.stack([img] * 3, axis=-1)  # CNN erwartet 3 Kanäle
            tensor = transform(img_rgb).unsqueeze(0).to(device)
            with torch.no_grad():
                feat = model(tensor).squeeze().cpu().numpy()
            feats.append(feat)
        except Exception:
            continue
    if not feats:
        return None
    return np.mean(feats, axis=0)


if HAS_TORCH and HAS_PYDICOM and TRAIN_IMG_DIR.exists():
    from torchvision import transforms as T
    from torchvision.models import resnet18, ResNet18_Weights

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Hinweis: Für eine Offline-Submission `weights=None` setzen und stattdessen lokal
    # zwischengespeicherte Gewichte laden (z. B. via `model.load_state_dict(torch.load(...))`
    # aus einem als Kaggle-Dataset eingebundenen .pth-File).
    try:
        weights = ResNet18_Weights.IMAGENET1K_V1
        cnn = resnet18(weights=weights)
        cnn.fc = torch.nn.Identity()  # letzten Klassifikations-Layer entfernen -> reine Features
        cnn.eval().to(device)

        transform = T.Compose([
            T.ToTensor(),
            T.Resize((224, 224)),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        # Demonstration an einer kleinen Stichprobe (aus Zeitgründen -- in einer vollen
        # Pipeline würde man das über alle Studies laufen lassen und cachen).
        demo_studies = train[ID_COL].head(5).tolist()
        image_features = {}
        for study_id in demo_studies:
            study_path = TRAIN_IMG_DIR / study_id
            if study_path.exists():
                feat = extract_study_image_features(study_path, cnn, transform, device)
                if feat is not None:
                    image_features[study_id] = feat

        print(f"Bildmerkmale extrahiert für {len(image_features)} Beispiel-Studies, "
              f"Feature-Dimension: {next(iter(image_features.values())).shape if image_features else '—'}")
        print("➡️  Diese Features könnten nun mit den TF-IDF-Textmerkmalen konkateniert und "
              "gemeinsam in ein Modell (z. B. LogisticRegression oder ein kleines MLP) gegeben werden "
              "-> multimodale Fusion (siehe Fazit).")
    except Exception as e:
        print(f"⚠️  Bild-Feature-Extraktion übersprungen (vermutlich kein Internetzugriff für "
              f"vortrainierte Gewichte in dieser Umgebung): {e}")
else:
    print("Überspringe Bild-Baseline-Demo (torch/pydicom fehlt oder Bildordner nicht gefunden).")


## 5. Finale Submission erzeugen

Wir trainieren die Text-Baseline nun auf **allen** verfügbaren Trainingsdaten (kein Train/Val-Split
mehr) und wenden sie auf `test.csv` an. Die Ausgabe wird **exakt** an das Schema von
`sample_submission.csv` angepasst:

- gleiche Spaltenreihenfolge,
- eine Zeile pro `StudyInstanceUID` aus `test.csv`,
- fehlende/nicht vorhersagbare Werte werden mit dem neutralen Default **0.5** aufgefüllt,
- Datei heißt exakt `submission.csv` und liegt im Arbeitsverzeichnis.


In [ ]:
# ---- Finales Modell auf allen Trainingsdaten ------------------------------------
X_full_text = train["Report"].fillna("")
y_full = train[TARGET_COLS].values

tfidf_final = TfidfVectorizer(max_features=20_000, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
X_full_tfidf = tfidf_final.fit_transform(X_full_text)

clf_final = OneVsRestClassifier(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED),
    n_jobs=-1,
)
clf_final.fit(X_full_tfidf, y_full)

print("Finales Modell auf allen", len(train), "Trainings-Studies trainiert.")


In [ ]:
# ---- Vorhersage auf Testdaten ----------------------------------------------------
test = pd.read_csv(TEST_CSV)
X_test_text = test["Report"].fillna("") if "Report" in test.columns else pd.Series([""] * len(test))
X_test_tfidf = tfidf_final.transform(X_test_text)

test_proba = clf_final.predict_proba(X_test_tfidf)

submission = pd.DataFrame(test_proba, columns=TARGET_COLS)
submission.insert(0, ID_COL, test[ID_COL].values)

# ---- Exakt an sample_submission.csv ausrichten ------------------------------------
# 1. Gleiche Spaltenreihenfolge erzwingen.
submission = submission[[ID_COL] + TARGET_COLS]

# 2. Sicherstellen, dass für jede erwartete StudyInstanceUID (aus sample_submission.csv)
#    auch tatsächlich eine Zeile existiert -- fehlende Studies werden mit dem neutralen
#    Default 0.5 aufgefüllt, damit die Submission niemals unvollständig ist.
submission = (
    sample_sub[[ID_COL]]
    .merge(submission, on=ID_COL, how="left")
)
submission[TARGET_COLS] = submission[TARGET_COLS].fillna(0.5)

# 3. Validierung vor dem Speichern.
assert list(submission.columns) == list(sample_sub.columns), "Spalten stimmen nicht mit sample_submission überein!"
assert len(submission) == len(sample_sub), "Zeilenanzahl stimmt nicht mit sample_submission überein!"
assert submission[TARGET_COLS].isna().sum().sum() == 0, "Es gibt noch fehlende Werte in der Submission!"
assert (submission[TARGET_COLS].values >= 0).all() and (submission[TARGET_COLS].values <= 1).all(), \
    "Alle Werte müssen zwischen 0 und 1 liegen!"

submission.to_csv("submission.csv", index=False)
print("✅ submission.csv gespeichert:", submission.shape)
submission.head()


## 6. Fazit & Ideen für Verbesserungen

Diese Baseline ist bewusst **einfach und robust** gehalten, um als gut nachvollziehbarer Startpunkt zu
dienen. Es gibt viel Potenzial, sie zu verbessern — hier ein paar konkrete Ansatzpunkte, sortiert grob
nach erwartetem Aufwand:

### Leicht umsetzbar
- **Iterative Multi-Label-Stratifikation** (z. B. `iterstrat`) statt zufälligem Split, damit seltene
  Labels (z. B. `Fracture`) in Train/Val gleichmäßiger verteilt sind.
- **Cross-Validation** (z. B. 5-fach) statt eines einzelnen Train/Val-Splits für stabilere
  Metriken und ein Ensemble aus Fold-Modellen.
- **Hyperparameter-Tuning** von TF-IDF (Vokabulargröße, n-gram-Range) und der logistischen Regression
  (Regularisierungsstärke `C`).
- **Andere Textmodelle**: z. B. `LinearSVC` mit kalibrierten Wahrscheinlichkeiten, Gradient Boosting
  auf TF-IDF-Features, oder ein einfaches neuronales Netz.

### Mittlerer Aufwand
- **Multimodale Fusion**: Bild- und Textmerkmale (wie im Bonus-Abschnitt skizziert) kombinieren, z. B.
  durch Konkatenation der Feature-Vektoren vor einem gemeinsamen Klassifikator, oder durch
  Late-Fusion (Mittelwert der Vorhersagen beider Modelle).
- **DICOM-Preprocessing verbessern**: Windowing/Normalisierung je nach Sequenztyp
  (`Fluid_Sensitive`, `Fat_Suppression`), Umgang mit unterschiedlichen **Transfer-Syntaxen** robust
  behandeln (z. B. komprimierte DICOMs, die spezielle Decoder benötigen), alle Slices statt nur des
  mittleren nutzen (z. B. via 2.5D- oder 3D-CNN).
- **Serien-Auswahl je nach `Anatomical_Plane`**: bestimmte Pathologien sind in bestimmten
  Ebenen (sagittal/koronal/axial) besser sichtbar — eine gezielte Serienauswahl pro Label könnte
  helfen.
- **Data Augmentation** für die Bildpipeline (Rotation, Flip, leichte Intensitätsvariation), um mit der
  begrenzten Datenmenge besser zu generalisieren.

### Höherer Aufwand / State-of-the-Art
- **Vortrainierte Vision-Language-Modelle** (z. B. medizinische CLIP-Varianten oder Modelle, die
  speziell auf Radiologie-Bild-Text-Paaren vortrainiert wurden) als starke Feature-Extraktoren nutzen
  oder fine-tunen.
- **3D-CNNs / Video-Transformer** über die gesamte Slice-Sequenz einer Serie, statt nur einen
  Einzel-Slice zu betrachten — Pathologien wie Menisken-Risse sind oft nur über mehrere Slices hinweg
  erkennbar.
- **Attention-basierte Aggregation** über Serien/Slices statt einfachem Mittelwert, damit das Modell
  lernt, welche Slices/Serien für welches Label am relevantesten sind.
- **Label-Korrelationen modellieren**: manche Pathologien treten häufig gemeinsam auf (z. B. Erguss und
  Synovitis) — ein Modell, das diese Korrelationen ausnutzt (z. B. Classifier-Chains oder ein
  gemeinsames Multi-Task-Netz statt 12 unabhängiger One-vs-Rest-Modelle), könnte davon profitieren.

### Praktische Hinweise für die Submission
- Denkt daran, alle vortrainierten Gewichte **vorab als Kaggle-Dataset** einzubinden, falls ihr Bild-
  oder Sprachmodelle nutzt — während der eigentlichen Submission-Ausführung ist **kein
  Internetzugriff** verfügbar.
- Testet die Laufzeit frühzeitig gegen das **9-Stunden-Limit**, insbesondere wenn ihr alle Slices statt
  nur einen repräsentativen Slice pro Serie verarbeitet.
- Validiert die Submission-Datei immer gegen `sample_submission.csv` (Spalten, Zeilenanzahl,
  Wertebereich) — genau wie im Code-Block oben.

---

**Wenn dir dieses Notebook geholfen hat, freue ich mich über ein Upvote! 🙂 Feedback, Forks und
Verbesserungsvorschläge sind herzlich willkommen.**
